In [1]:
import sys 
import numpy as np
import cv2
import tensorflow as tf
from tqdm import tqdm
import sleap

In [2]:
def video_loader(video_path, target_size=None, load_as_tensor=False):
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if target_size:
            gray = cv2.resize(gray, target_size)  # (width, height)
        frames.append(gray)
    
    cap.release()
    video_array = np.array(frames)  # Shape: (num_frames, height, width)
    
    if load_as_tensor:
        video_tensor = tf.constant(video_array, dtype=tf.float16) / 255.0
        video_tensor = tf.expand_dims(video_tensor, axis=-1)  # Add channel dim
        return video_tensor
    else:
        return video_array / 255.0

In [3]:
full_video_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_15min_reencoded_v2.mp4'
clipped_video_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_high_res_5min_track_reencoded_0.mp4'

full_video = video_loader(full_video_path)
clipped_video = video_loader(clipped_video_path)

In [4]:
pad_inx = 150 * 50 * 60
start_search_idx = 0
print(f'pad_inx: {pad_inx}')
print(f'full_video.shape: {full_video.shape}')
print(f'clipped_video.shape: {clipped_video.shape}')

pad_inx: 450000
full_video.shape: (135000, 170, 174)
clipped_video.shape: (9000, 170, 174)


In [16]:
clipped_first_frame = clipped_video[0]
clipped_last_frame = clipped_video[-1]
first_match_idx = None
last_match_idx = None
for i in tqdm(range(start_search_idx, len(full_video))):
    if np.allclose(full_video[i], clipped_first_frame, atol=0.1):
        print(i)
        match_idx = i
    elif np.allclose(full_video[i], clipped_last_frame, atol=0.12):
        print(i)
        last_match_idx = i
        # break

# print(f'match_idx: {match_idx + pad_inx}')

 23%|██▎       | 30424/135000 [00:07<00:26, 3938.76it/s]

29835
29836
29837
29838
29840
29841
29842
29843
29844
29845
29846
29850
29851
29852
29853
29854
29855
29856
29864
29865


 29%|██▉       | 39461/135000 [00:10<00:24, 3909.28it/s]

38850


100%|██████████| 135000/135000 [00:34<00:00, 3892.39it/s]


In [10]:
29835/150/60

3.315

In [17]:
38850 - 9000

29850

In [3]:
pad_inx = 150 * 50 * 60
start_idx = 29850 + pad_inx
end_idx = 38850 + pad_inx
print(f'start index: {start_idx}')
print(f'end index: {end_idx}')

start index: 479850
end index: 488850


In [4]:
indices_switch = [12, 9, 10, 8, 4, 14, 15, 16, 0, 1, 2, 5, 3, 13, 11, 7, 6]

In [5]:
a1_dataset_path = '/home/mingxiao/Desktop/jellyfish/label/animal_1_labels.v002.slp'
a1_dataset = sleap.load_file(a1_dataset_path)
print(a1_dataset)

Labels(labeled_frames=8089, videos=1, skeletons=1, tracks=0)


In [6]:
c1_dataset_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/manual_5min_c0.slp'
c1_dataset = sleap.load_file(c1_dataset_path)
print(c1_dataset)

Labels(labeled_frames=9000, videos=1, skeletons=1, tracks=17)


In [7]:
def get_coords_single_model(lf):
    for inst in lf.instances:
        if not isinstance(inst, sleap.instance.PredictedInstance):
            return inst.points_array[1:]
    raise ValueError('No labeled instance found')

def get_coords_multi_model(lf):
    coords = np.zeros((17, 2))
    if len(lf.instances) != 17:
        print(len(lf.instances))
    for inst in lf.instances:
        # if not isinstance(inst, sleap.instance.PredictedInstance):
        if True:
            idx = int(inst.track.name[6:])
            coords[idx] = inst.points_array[0]
    if not np.all(coords != 0): 
        print('some points are missing')
        return
    return coords

In [8]:
import sleap.instance

labeled_indices = []
labeled_coords = []
lb_cnt = 0
for lf in a1_dataset.labeled_frames:
    if len(lf.instances) == 2:
        lb_cnt += 1
        labeled_indices.append(lf.frame_idx)
        labeled_coords.append(get_coords_single_model(lf))
    elif len(lf.instances) == 1 and not isinstance(lf.instances[0], sleap.instance.PredictedInstance):
        lb_cnt += 1
        labeled_indices.append(lf.frame_idx)
        labeled_coords.append(get_coords_single_model(lf))
labeled_coords_arr = np.array(labeled_coords)
print(labeled_coords_arr.shape)
print(f'lb_cnt: {lb_cnt}')

for lf in tqdm(c1_dataset.labeled_frames[:2500]):
    curr_frame_coords = get_coords_multi_model(lf)
    if curr_frame_coords is None:
        continue
    padded_idx = lf.frame_idx + start_idx
    if padded_idx in labeled_indices:
        corresponding_idx = labeled_indices.index(padded_idx)
        coords_1 = labeled_coords_arr[corresponding_idx]
        if not np.allclose(curr_frame_coords, coords_1, atol=20):
            labeled_coords_arr[corresponding_idx] = curr_frame_coords
            print(f'coordinates do not match at frame {lf.frame_idx}')
    else:
        labeled_indices.append(padded_idx)
        labeled_coords.append(curr_frame_coords)
        
labeled_coords_arr = np.array(labeled_coords)
print(labeled_coords_arr.shape)

sorted_idx = np.argsort(labeled_indices)
labeled_indices = np.array(labeled_indices)[sorted_idx]
labeled_coords_arr = labeled_coords_arr[sorted_idx]

(1653, 17, 2)
lb_cnt: 1653


 20%|██        | 508/2500 [00:00<00:01, 1004.67it/s]

coordinates do not match at frame 276


 92%|█████████▏| 2301/2500 [00:02<00:00, 970.60it/s]

16
some points are missing
16
some points are missing


100%|██████████| 2500/2500 [00:02<00:00, 984.62it/s]

16
some points are missing
(4149, 17, 2)


In [78]:
len(labeled_indices)

4149

In [79]:
sorted_idx = np.argsort(labeled_indices)
labeled_indices = np.array(labeled_indices)[sorted_idx]
labeled_coords_arr = labeled_coords_arr[sorted_idx]
print(labeled_coords_arr.shape)

[      0    3240    6480 ... 3230280 3233520 3236760]
(4149, 17, 2)


In [62]:
start_idx

479850

In [64]:
lst = [1, 2, 3]
lst.index(2)

1

In [67]:
for lf in tqdm(c1_dataset.labeled_frames[:2500]):
    curr_frame_coords = get_coords_multi_model(lf)
    padded_idx = lf.frame_idx + start_idx
    if padded_idx in labeled_indices:
        coords_1 = labeled_coords_arr[labeled_indices.index(padded_idx)]
        assert np.allclose(curr_frame_coords, coords_1, atol=20), f'coordinates do not match at frame {lf.frame_idx}'
    else:
        labeled_indices.append(padded_idx)
        labeled_coords.append(curr_frame_coords)
        
labeled_coords_arr = np.array(labeled_coords)
print(labeled_coords_arr.shape)


 11%|█         | 276/2500 [00:00<00:02, 944.42it/s]


AssertionError: coordinates do not match at frame 276